# 04 - Deep Learning model: DistilBERT (fine-tuning)
**NLP Deliverable 2 — modern ML approach #2**

We fine-tune the pre-trained transformer **`distilbert-base-cased`** for token
classification with Hugging Face `transformers`. Sub-word tokens are aligned back to
word-level labels for evaluation.

Saved artifacts (used by `05_reproduce_results.ipynb`):
`fitted_models/distilbert_ner/` (model + tokenizer + label maps).

> **Designed for Google Colab with a GPU** (full training). On CPU the notebook
> automatically switches to a small *fast* run (subset + 1 epoch) so it still completes
> and saves a working model; set `FAST = False` on a GPU for the full configuration.

In [ ]:
# === Setup (works locally and on Google Colab) ===
import sys, os
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install -q transformers datasets evaluate seqeval
    DATA_DIR = '.'
else:
    DATA_DIR = '../nlp_d2_data'
FITTED_DIR = 'fitted_models'
os.makedirs(FITTED_DIR, exist_ok=True)

import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
import torch
from datasets import Dataset
from transformers import (AutoTokenizer, AutoModelForTokenClassification,
                          TrainingArguments, Trainer, DataCollatorForTokenClassification)
import evaluate
from sklearn.metrics import f1_score, accuracy_score, confusion_matrix
sns.set_theme(style='whitegrid')

HAS_GPU = torch.cuda.is_available()
# Full run only makes sense on a GPU; CPU falls back to a quick smoke run.
FAST = not HAS_GPU
N_TRAIN = 4000 if FAST else None     # subset size in FAST mode (None = all)
N_EVAL  = 2000 if FAST else None
EPOCHS  = 1 if FAST else 3
print(f'GPU available: {HAS_GPU} | FAST mode: {FAST} | epochs: {EPOCHS}')

## 1. Load data and build label maps

In [ ]:
def load_grouped(path):
    df = pd.read_csv(path)
    df['words'] = df['words'].astype(str)
    df['sentence_id'] = df['sentence_id'].astype(int)
    g = df.groupby('sentence_id', sort=True).agg({'words': list, 'tags': list})
    return g.reset_index(drop=True)

train_g = load_grouped(os.path.join(DATA_DIR, 'train_data_ner.csv'))
test_g  = load_grouped(os.path.join(DATA_DIR, 'test_data_ner.csv'))

label_list = sorted({t for row in train_g['tags'] for t in row})
label2id = {l: i for i, l in enumerate(label_list)}
id2label = {i: l for l, i in label2id.items()}
print(f'{len(label_list)} labels:', label_list)

## 2. Tokenize and align sub-word labels

In [ ]:
model_ckpt = 'distilbert-base-cased'
tokenizer = AutoTokenizer.from_pretrained(model_ckpt)

def make_ds(g, n=None):
    if n is not None:
        g = g.iloc[:n]
    return Dataset.from_dict({'words': g['words'].tolist(), 'tags': g['tags'].tolist()})

def tokenize_align(examples):
    tok = tokenizer(examples['words'], truncation=True, is_split_into_words=True)
    all_labels = []
    for i, tags in enumerate(examples['tags']):
        word_ids = tok.word_ids(batch_index=i)
        prev, ids = None, []
        for wid in word_ids:
            if wid is None:
                ids.append(-100)
            elif wid != prev:                 # label only the first sub-token
                ids.append(label2id[tags[wid]])
            else:
                ids.append(-100)
            prev = wid
        all_labels.append(ids)
    tok['labels'] = all_labels
    return tok

train_ds = make_ds(train_g, N_TRAIN).map(tokenize_align, batched=True)
eval_ds  = make_ds(test_g,  N_EVAL ).map(tokenize_align, batched=True)
print('tokenized train:', len(train_ds), '| eval:', len(eval_ds))

## 3. Fine-tune

In [ ]:
model = AutoModelForTokenClassification.from_pretrained(
    model_ckpt, num_labels=len(label_list), id2label=id2label, label2id=label2id)

seqeval = evaluate.load('seqeval')
def compute_metrics(p):
    preds, labels = p
    preds = np.argmax(preds, axis=2)
    true_pred = [[label_list[pp] for pp, ll in zip(pr, la) if ll != -100]
                 for pr, la in zip(preds, labels)]
    true_lab  = [[label_list[ll] for pp, ll in zip(pr, la) if ll != -100]
                 for pr, la in zip(preds, labels)]
    r = seqeval.compute(predictions=true_pred, references=true_lab)
    return {'precision': r['overall_precision'], 'recall': r['overall_recall'],
            'f1': r['overall_f1'], 'accuracy': r['overall_accuracy']}

args = TrainingArguments(
    output_dir='distilbert-ner-ckpt',
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=EPOCHS,
    weight_decay=0.01,
    logging_steps=100,
    report_to='none',
    save_strategy='no',
)
collator = DataCollatorForTokenClassification(tokenizer=tokenizer)
trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=eval_ds,
                  tokenizer=tokenizer, data_collator=collator, compute_metrics=compute_metrics)
trainer.train()
print(trainer.evaluate())

In [ ]:
# Persist model + tokenizer for reproduce_results.ipynb
save_dir = os.path.join(FITTED_DIR, 'distilbert_ner')
model.save_pretrained(save_dir)
tokenizer.save_pretrained(save_dir)
print('Saved', save_dir)

## 4. Word-level evaluation (align sub-words back to words)

In [ ]:
device = model.device

@torch.no_grad()
def predict_words(list_of_words, batch_size=32):
    model.eval()
    out_tags = []
    for start in range(0, len(list_of_words), batch_size):
        batch = list_of_words[start:start+batch_size]
        enc = tokenizer(batch, truncation=True, is_split_into_words=True,
                        padding=True, return_tensors='pt').to(device)
        logits = model(**enc).logits
        pred = logits.argmax(-1).cpu().numpy()
        for i in range(len(batch)):
            word_ids = enc.word_ids(batch_index=i)
            prev, tags = None, []
            for j, wid in enumerate(word_ids):
                if wid is None or wid == prev:
                    prev = wid; continue
                tags.append(id2label[int(pred[i][j])])
                prev = wid
            # safety: pad/truncate to number of words
            words = batch[i]
            if len(tags) < len(words):
                tags += ['O']*(len(words)-len(tags))
            out_tags.append(tags[:len(words)])
    return out_tags

def flatten(s): return [x for sub in s for x in sub]
def non_o_accuracy(yt, yp):
    yt, yp = flatten(yt), flatten(yp)
    idx = [k for k, t in enumerate(yt) if t != 'O']
    return accuracy_score([yt[k] for k in idx], [yp[k] for k in idx])
def weighted_f1(yt, yp):
    yt, yp = flatten(yt), flatten(yp)
    labels = sorted(set(yt) - {'O'})
    return f1_score(yt, yp, average='weighted', labels=labels, zero_division=0)

eval_slice = test_g.iloc[:N_EVAL] if N_EVAL else test_g
y_true = eval_slice['tags'].tolist()
y_pred = predict_words(eval_slice['words'].tolist())
print(f'Test non-O accuracy: {non_o_accuracy(y_true, y_pred):.4f} | '
      f'weighted F1: {weighted_f1(y_true, y_pred):.4f}'
      + ('   (FAST subset)' if FAST else ''))

In [ ]:
yt, yp = flatten(y_true), flatten(y_pred)
labels = sorted(set(yt) - {'O'})
cm = confusion_matrix(yt, yp, labels=labels)
plt.figure(figsize=(9,7))
sns.heatmap(cm, annot=True, fmt='d', cmap='Purples', xticklabels=labels, yticklabels=labels)
plt.title('DistilBERT — test confusion matrix (excluding O)')
plt.ylabel('True'); plt.xlabel('Predicted'); plt.show()

## 5. TINY TEST

In [ ]:
def load_grouped_words(path):
    g = load_grouped(path)
    return g['words'].tolist(), g['tags'].tolist()

tiny_words, tiny_tags = load_grouped_words(os.path.join(DATA_DIR, 'tiny_test.csv'))
tiny_pred = predict_words(tiny_words)
print('=== TINY TEST — DistilBERT ===')
for ws, ps in zip(tiny_words, tiny_pred):
    print(' '.join(f'{w}/{t}' for w, t in zip(ws, ps)))
print(f'\nTiny-test accuracy (non-O): {non_o_accuracy(tiny_tags, tiny_pred):.4f}')

## 6. Notes
- This was run in **FAST mode** if no GPU was detected (subset + 1 epoch) just to verify
  the pipeline and save a working model. For the reported results, run on **Colab GPU**
  with `FAST = False` (full data, 3 epochs) — DistilBERT is expected to be the strongest
  model thanks to pre-trained contextual representations and sub-word handling of OOV words.